# 🎓 Publishable Master Corpus NMT Architecture (Zero Data Leakage)
### Rigorous Multilingual Neural Machine Translation (English ↔ Kiswahili ↔ Ekegusii)

This production-grade notebook implements the **Master Corpus Database Architecture**:

--- 
### 🌟 Scientific Rigor & Engineering Principles:
1. **Master Sentence Corpus Database**: Unified **49,277 multilingual concepts** with unique Concept IDs (`concept_id`).
2. **Separate Master Lexical Corpus**: Isolated **268 dictionary entries** to prevent vocabulary leakage into evaluation sets.
3. **Single Master Split (80% / 10% / 10%)**: Split ONCE before deriving any training sets. Zero concept leakage across Monolingual, Bilingual, Trilingual, or Domain adaptation training.
4. **High-Capacity LoRA ($r=32$, $\alpha=64$)**: Targets Attention and Feed-Forward projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `fc1`, `fc2`).
5. **Orthography Normalizer & Repetition Penalty Guard**: Applies regex rule-based spelling standardization and `repetition_penalty=1.25` during generation decoding.

## 1. Environment Setup & GPU Memory Guard

In [ ]:
# Install latest dependencies
%pip install -U \
transformers \
peft \
datasets \
evaluate \
sacrebleu \
accelerate \
sentencepiece \
bitsandbytes \
matplotlib \
pandas \
numpy \
scikit-learn \
seaborn

import torch
import transformers
import peft
import datasets
import evaluate
import os
import sys
import glob
import pandas as pd
import numpy as np
import re
import gc

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

clear_gpu_memory()

print('=== GPU Hardware Info ===')
print('PyTorch Version:', torch.__version__)
print('Transformers Version:', transformers.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('Total VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: Running on CPU.')

## 2. Load Master Corpus Database Splits (Zero Data Leakage)
Loads the pre-split Master Sentence Corpus (`master_train.csv`, `master_val.csv`, `master_test.csv`) and derived sub-views.

In [ ]:
def normalize_ekegusii_orthography(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = re.sub(r'\beserekari\b', 'eserikari', text, flags=re.IGNORECASE)
    text = re.sub(r'\begeombe\b', 'ekeombe', text, flags=re.IGNORECASE)
    text = re.sub(r'\bkovatania\b', 'kobwatania', text, flags=re.IGNORECASE)
    text = re.sub(r'\bchinyomba\b', 'chinyomba', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Locate Master Corpus Directory
master_dir = os.path.join('data', 'master_corpus', 'splits')
if not os.path.exists(master_dir):
    master_dir = os.path.join('..', 'data', 'master_corpus', 'splits')

train_master_path = os.path.join(master_dir, 'master_train.csv')
val_master_path = os.path.join(master_dir, 'master_val.csv')
test_master_path = os.path.join(master_dir, 'master_test.csv')

master_train = pd.read_csv(train_master_path)
master_val = pd.read_csv(val_master_path)
master_test = pd.read_csv(test_master_path)

print(f'=== MASTER CORPUS DATABASE LOADED ===')
print(f' -> Master Train Set      : {len(master_train)} concepts')
print(f' -> Master Validation Set : {len(master_val)} concepts')
print(f' -> Master Test Set       : {len(master_test)} concepts')

# Verify Zero Data Leakage (Intersection of Concept IDs must be 0)
train_ids = set(master_train['concept_id'])
val_ids = set(master_val['concept_id'])
test_ids = set(master_test['concept_id'])

leakage = (train_ids & test_ids) | (train_ids & val_ids) | (val_ids & test_ids)
print(f' -> Data Leakage Verification: {len(leakage)} overlapping concepts (MUST BE 0!)')

## 3. Derive Parallel Training Sets & Tokenizer Preprocessing
Derives `ENG-EKE`, `SWA-EKE`, and `Trilingual` parallel pairs strictly from `master_train.csv`.

In [ ]:
def build_bidirectional_pairs(df, src_col, tgt_col):
    sub = df.dropna(subset=[src_col, tgt_col])
    forward = pd.DataFrame({'src': sub[src_col], 'tgt': sub[tgt_col], 'src_lang': src_col, 'tgt_lang': tgt_col})
    backward = pd.DataFrame({'src': sub[tgt_col], 'tgt': sub[src_col], 'src_lang': tgt_col, 'tgt_lang': src_col})
    return pd.concat([forward, backward], ignore_index=True).dropna().drop_duplicates().reset_index(drop=True)

# Derive Training Subsets
bi_eng_eke = build_bidirectional_pairs(master_train, 'English', 'Ekegusii')
bi_swa_eke = build_bidirectional_pairs(master_train, 'Kiswahili', 'Ekegusii')
train_bi_all = pd.concat([bi_eng_eke, bi_swa_eke], ignore_index=True).drop_duplicates().reset_index(drop=True)

# Derive Evaluation Subsets
val_psa_sub = master_val[master_val['source'] == 'PSA']
val_eval_pairs = build_bidirectional_pairs(val_psa_sub, 'English', 'Ekegusii')

test_psa_sub = master_test[master_test['source'] == 'PSA']
test_eval_pairs = build_bidirectional_pairs(test_psa_sub, 'English', 'Ekegusii')

print(f'[OK] Derived Bidirectional Training Pairs: {len(train_bi_all)}')
print(f'[OK] Derived Evaluation Pairs: Val={len(val_eval_pairs)} | Test={len(test_eval_pairs)}')

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'facebook/nllb-200-distilled-600M'
LANG_TAGS = {'English': 'eng_Latn', 'Kiswahili': 'swh_Latn', 'Ekegusii': 'swh_Latn'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'fc1', 'fc2']
)

def preprocess_master_nmt(examples):
    src_texts = [str(x) for x in examples['src']]
    tgt_texts = [str(x) for x in examples['tgt']]
    src_langs = examples['src_lang']
    tgt_langs = examples['tgt_lang']
    
    model_inputs = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for s_text, t_text, s_l, t_l in zip(src_texts, tgt_texts, src_langs, tgt_langs):
        tokenizer.src_lang = LANG_TAGS.get(s_l, 'eng_Latn')
        tokenizer.tgt_lang = LANG_TAGS.get(t_l, 'swh_Latn')
        inp = tokenizer(s_text, max_length=128, truncation=True, padding=False)
        lbl = tokenizer(text_target=t_text, max_length=128, truncation=True, padding=False)
        model_inputs['input_ids'].append(inp['input_ids'])
        model_inputs['attention_mask'].append(inp['attention_mask'])
        model_inputs['labels'].append(lbl['input_ids'])
    return model_inputs

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [normalize_ekegusii_orthography(pred.strip()) for pred in decoded_preds]
    decoded_labels = [[normalize_ekegusii_orthography(label.strip())] for label in decoded_labels]
    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': bleu['score'], 'chrf': chrf['score']}

print('[OK] Tokenizer & High-Capacity Preprocessing Configured.')

## 4. Fine-Tuning Execution on Master Train Split
Trains Meta NLLB-200 with LoRA ($r=32$) on the Master Train Split dataset.

In [ ]:
print('=== RUNNING MASTER CORPUS NMT FINE-TUNING ===')
clear_gpu_memory()

model_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=model_dtype).to(device)
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

ds_train = datasets.Dataset.from_pandas(train_bi_all.sample(min(20000, len(train_bi_all)))).map(preprocess_master_nmt, batched=True)
ds_val = datasets.Dataset.from_pandas(val_eval_pairs).map(preprocess_master_nmt, batched=True)

args = Seq2SeqTrainingArguments(
    output_dir='./output_master_corpus',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='chrf',
    greater_is_better=True,
    learning_rate=4e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    predict_with_generate=True,
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics
)

trainer.train()
clear_gpu_memory()
print('[OK] Master Corpus Fine-Tuning Completed Successfully.')

## 5. Benchmark Evaluation on Held-Out Master Test Split
Evaluates SacreBLEU and chrF++ scores on the 100% leak-proof held-out test split (`derived_test_psa.csv`).

In [ ]:
clear_gpu_memory()
ds_test = datasets.Dataset.from_pandas(test_eval_pairs).map(preprocess_master_nmt, batched=True)
eval_args = Seq2SeqTrainingArguments(output_dir='./output_eval_master', per_device_eval_batch_size=32, predict_with_generate=True, bf16=torch.cuda.is_bf16_supported(), report_to='none')
eval_trainer = Seq2SeqTrainer(model=model, args=eval_args, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model), compute_metrics=compute_metrics)
final_metrics = eval_trainer.evaluate(ds_test)
clear_gpu_memory()

bleu_score = final_metrics.get('eval_bleu', 0.0)
chrf_score = final_metrics.get('eval_chrf', 0.0)

print('=== 🏆 MASTER CORPUS BENCHMARK EVALUATION RESULTS ===')
print(f' -> Final SacreBLEU Score : {bleu_score:.2f}')
print(f' -> Final chrF++ Score    : {chrf_score:.2f}')

# Generate Live Qualitative Translations
print('\n=== 💬 QUALITATIVE TRANSLATION PREDICTIONS ===')
sample_sources = test_psa_sub['English'].iloc[:5].tolist()
sample_references = test_psa_sub['Ekegusii'].iloc[:5].tolist()

for i, (src, ref) in enumerate(zip(sample_sources, sample_references), 1):
    tokenizer.src_lang = 'eng_Latn'
    tokenizer.tgt_lang = 'swh_Latn'
    inputs = tokenizer(src, return_tensors='pt', max_length=128, truncation=True).to(device)
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs, 
            max_length=128, 
            num_beams=4, 
            repetition_penalty=1.25, 
            no_repeat_ngram_size=3
        )
    pred_raw = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    pred_normalized = normalize_ekegusii_orthography(pred_raw)
    print(f'\nSample {i}:')
    print(f'  [SOURCE English]   : {src}')
    print(f'  [REFERENCE Target] : {ref}')
    print(f'  [MODEL GENERATED]  : {pred_normalized}')

## 6. Permanent Model Exporter & Saver
Saves the fine-tuned LoRA weights and tokenizer permanently into `models/master_corpus_nmt_model/`.

In [ ]:
save_directory = './models/master_corpus_nmt_model'
os.makedirs(save_directory, exist_ok=True)
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f'=== 💾 MODEL SAVED PERMANENTLY ===')
print(f'[OK] Fine-Tuned Model Weights & Tokenizer Saved to: "{save_directory}"')
print('You can now load this model in production using AutoModelForSeq2SeqLM.from_pretrained("./models/master_corpus_nmt_model")!')